# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `YOUTUBE_API_KEY` | **初回だけ**入力。Google Cloud Console で取得（**無料**） |
| `CLAUDE_API_KEY` | **初回だけ**入力。console.anthropic.com で取得（有料・`sk-ant-`で始まる） |
| `PEXELS_API_KEY` | **初回だけ**入力。pexels.com/api で取得（**無料**） |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 💰 Claude APIの費用目安
| 使い方 | 費用 |
|---|---|
| 1回10本生成 | 約5円 |
| 毎日10本 × 30日 | 約160円/月 |

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約5〜10分 |
| 5本 | 約25〜40分 |
| 10本 | 約50〜80分 |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: ここだけ変える（毎回）                        ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキー ──────────────────────────────────────────────
YOUTUBE_API_KEY = ''   # ← Google Cloud Console で取得（無料）
CLAUDE_API_KEY  = ''   # ← console.anthropic.com で取得（sk-ant-で始まる）
PEXELS_API_KEY  = ''   # ← pexels.com/api で取得（無料）

# ── テーマ（毎回変える） ─────────────────────────────────
THEME = 'ダイエット'
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── 生成する動画の本数 ───────────────────────────────────
VIDEO_COUNT = 10        # ← 1〜10本

# ── 動画の長さ ───────────────────────────────────────────
DURATION = 45           # ← 目標秒数（15〜60）。内容に応じてシーン数・尺を自動調整

# ── 画像スタイル ─────────────────────────────────────────
IMAGE_STYLE = 'realistic'    # 写真そのまま
# IMAGE_STYLE = 'anime'        # アニメ風
# IMAGE_STYLE = 'manga'        # 漫画風（白黒）
# IMAGE_STYLE = 'illustration' # イラスト風

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

if not YOUTUBE_API_KEY.strip():
    raise ValueError('❌ YOUTUBE_API_KEY を入力してください（Google Cloud Console で取得）')
if not CLAUDE_API_KEY.strip():
    raise ValueError('❌ CLAUDE_API_KEY を入力してください（console.anthropic.com で取得）')
if not PEXELS_API_KEY.strip():
    raise ValueError('❌ PEXELS_API_KEY を入力してください（pexels.com/api で取得）')

import os
os.makedirs('/content/output', exist_ok=True)

print('設定内容を確認します...')
print(f'  テーマ      : {THEME}')
print(f'  生成本数    : {VIDEO_COUNT}本')
print(f'  目標秒数    : {DURATION}秒（自動調整）')
print(f'  画像スタイル: {IMAGE_STYLE}')
print()
print('✅ 設定完了！')

In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg fonts-noto-cjk p7zip-full
print('✅ ツールのインストール完了')

# ── VOICEVOX Engine セットアップ（完全無料・ローカル動作）──
import subprocess as _sp, os as _os, time as _time
import requests as _req
VV_PORT = 50021
VV_URL  = f'http://127.0.0.1:{VV_PORT}'
VV_DIR  = '/content/voicevox_engine'
VV_SPEAKER = 3  # ずんだもん（ノーマル）
VV_AVAILABLE = False

if not _os.path.exists(f'{VV_DIR}/run'):
    print('🔊 VOICEVOXエンジンをダウンロード中（初回のみ・数分かかります）...')
    try:
        _rel = _req.get(
            'https://api.github.com/repos/VOICEVOX/voicevox_engine/releases/latest',
            timeout=30).json()
        # v0.20以降は .7z.001 形式（Linux x64 CPU版）
        _asset = next(
            (a for a in _rel['assets']
             if 'linux-cpu-x64' in a['name'] and a['name'].endswith('.7z.001')),
            None
        )
        if _asset is None:
            raise RuntimeError(f'Linux CPU x64アセットが見つかりません。利用可能: {[a["name"] for a in _rel["assets"]]}')
        _fn = f'/tmp/{_asset["name"]}'
        print(f'  ダウンロード: {_asset["name"]}')
        _sp.run(['wget', '-q', '--show-progress', '-O', _fn,
                 _asset['browser_download_url']], check=True)
        _os.makedirs(VV_DIR, exist_ok=True)
        _sp.run(['7z', 'x', _fn, f'-o{VV_DIR}', '-y'],
                check=True, capture_output=True)
        _os.remove(_fn)
        # 7z展開でサブディレクトリが作られた場合は1段上げる
        _subdirs = [d for d in _os.listdir(VV_DIR)
                    if _os.path.isdir(f'{VV_DIR}/{d}') and not d.startswith('.')]
        if _subdirs and not _os.path.exists(f'{VV_DIR}/run'):
            _sub = f'{VV_DIR}/{_subdirs[0]}'
            for _item in _os.listdir(_sub):
                _sp.run(['mv', f'{_sub}/{_item}', VV_DIR], check=True)
            _os.rmdir(_sub)
        _sp.run(['chmod', '+x', f'{VV_DIR}/run'], check=False)
        print('✅ VOICEVOXダウンロード・展開完了')
    except Exception as _e:
        print(f'⚠️ VOICEVOXダウンロード失敗（gTTSで代替します）: {_e}')

if _os.path.exists(f'{VV_DIR}/run'):
    _sp.Popen(
        [f'{VV_DIR}/run', '--host', '127.0.0.1', '--port', str(VV_PORT)],
        stdout=_sp.DEVNULL, stderr=_sp.DEVNULL
    )
    print('🔊 VOICEVOXエンジン起動中（最大60秒）...')
    for _i in range(60):
        _time.sleep(1)
        try:
            if _req.get(f'{VV_URL}/version', timeout=2).status_code == 200:
                VV_AVAILABLE = True
                print(f'✅ VOICEVOX起動完了（{_i+1}秒）→ 高品質音声モード')
                break
        except:
            pass
    if not VV_AVAILABLE:
        print('⚠️ VOICEVOX起動タイムアウト → gTTSにフォールバック')
else:
    print('⚠️ VOICEVOXが見つかりません → gTTSで音声生成します')


In [ ]:
# 【自動】① YouTubeトレンド取得 → ② 人気動画の詳細分析 → ③ Claude AI分析 → ④ 切り口生成

import requests as _req
import json, re

def fetch_youtube_trends(theme):
    """2パス検索: ①直近90日トレンド(relevance) + ②全期間人気(viewCount) → 重複排除"""
    import datetime
    after = (datetime.datetime.utcnow() - datetime.timedelta(days=90)).strftime('%Y-%m-%dT%H:%M:%SZ')
    base = {
        'part': 'snippet', 'q': f'{theme} shorts', 'type': 'video',
        'regionCode': 'JP', 'relevanceLanguage': 'ja',
        'key': YOUTUBE_API_KEY, 'videoDuration': 'short',
    }
    passes = [
        {**base, 'order': 'relevance', 'maxResults': 15, 'publishedAfter': after},  # 最新トレンド
        {**base, 'order': 'viewCount',  'maxResults': 10},                          # 歴代人気パターン
    ]
    results, seen = [], set()
    for params in passes:
        try:
            r = _req.get('https://www.googleapis.com/youtube/v3/search', params=params, timeout=15)
            if r.status_code != 200:
                print(f'  ⚠ YouTube検索API: {r.status_code}')
                continue
            for item in r.json().get('items', []):
                vid = item['id'].get('videoId', '')
                if vid and vid not in seen:
                    seen.add(vid)
                    results.append({
                        'id': vid,
                        'title':   item['snippet'].get('title', ''),
                        'channel': item['snippet'].get('channelTitle', ''),
                    })
        except Exception as e:
            print(f'  ⚠ YouTube API失敗: {e}')
    return results

def fetch_video_stats(video_ids):
    """再生数・いいね・コメ・タグ・尺を取得。60秒超えはShortsではないので除外。"""
    import re as _re
    if not video_ids: return []
    url = 'https://www.googleapis.com/youtube/v3/videos'
    params = {
        'part': 'snippet,statistics,contentDetails',  # contentDetails で尺チェック
        'id': ','.join(video_ids[:20]),               # 最大20件
        'key': YOUTUBE_API_KEY,
    }
    def _parse_duration(iso):
        """PT1M30S → 90秒 に変換"""
        m = _re.search(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso or '')
        if not m: return 999
        h, mn, s = (int(x or 0) for x in m.groups())
        return h*3600 + mn*60 + s
    try:
        r = _req.get(url, params=params, timeout=15)
        if r.status_code == 200:
            results = []
            for item in r.json().get('items', []):
                snip  = item.get('snippet', {})
                stats = item.get('statistics', {})
                dur   = _parse_duration(item.get('contentDetails', {}).get('duration', ''))
                if dur > 60: continue  # 60秒超えはShortsではない
                results.append({
                    'title':       snip.get('title', ''),
                    'description': snip.get('description', '')[:200],
                    'tags':        snip.get('tags', [])[:8],
                    'duration_s':  dur,
                    'views':       int(stats.get('viewCount', 0)),
                    'likes':       int(stats.get('likeCount', 0)),
                    'comments':    int(stats.get('commentCount', 0)),
                })
            return sorted(results, key=lambda x: x['views'], reverse=True)
        else:
            print(f'  ⚠ YouTube統計API: {r.status_code}')
            return []
    except Exception as e:
        print(f'  ⚠ 統計取得失敗: {e}')
        return []

CLAUDE_URL = 'https://api.anthropic.com/v1/messages'

def call_ai(prompt, tokens=4096):
    res = _req.post(
        CLAUDE_URL,
        headers={'x-api-key': CLAUDE_API_KEY,
                 'anthropic-version': '2023-06-01',
                 'content-type': 'application/json'},
        json={'model': 'claude-haiku-4-5-20251001',
              'max_tokens': tokens,
              'messages': [{'role': 'user', 'content': prompt}]},
        timeout=120
    )
    if res.status_code != 200:
        err = res.text[:300].replace(CLAUDE_API_KEY, '***')
        raise RuntimeError(f'Claude APIエラー ({res.status_code}): {err}')
    return res.json()['content'][0]['text']

def clean_title(t):
    t = re.sub(r'^#+\s*', '', t)
    t = re.sub(r'\*+', '', t)
    t = re.sub(r'^\d+[\.\)]\s*', '', t)
    return t.strip()

# ── ① YouTube人気動画を検索 ──────────────────────────────────
print(f'📺 YouTubeで「{THEME}」の人気動画を検索中...')
yt_basic = fetch_youtube_trends(THEME)

if yt_basic:
    print(f'  ✓ {len(yt_basic)}件の動画を発見')
    video_ids = [v['id'] for v in yt_basic if v['id']]

    # ── ② 再生数・いいね数・タグを詳細取得 ──────────────────
    print(f'  📊 動画の詳細データを取得中...')
    yt_stats = fetch_video_stats(video_ids)

    if yt_stats:
        print(f'  ✓ 統計データ取得完了（上位{len(yt_stats)}本）')
        print(f'\n  🔥 最も再生されている動画:')
        for v in yt_stats[:3]:
            views_str = f"{v['views']:,}" if v['views'] > 0 else '非公開'
            print(f'    再生:{views_str} │ {v["title"][:40]}')

        # 分析用テキスト（再生数・タグ・説明文を含む）
        analysis_lines = []
        for i, v in enumerate(yt_stats[:15]):
            tags_str = '・'.join(v['tags'][:6]) if v['tags'] else 'タグなし'
            dur_str = f"{v.get('duration_s',0)}秒" if v.get('duration_s') else ''
            analysis_lines.append(
                f'{i+1}. 「{v["title"]}」{dur_str}\n'
                f'   再生:{v["views"]:,} / いいね:{v["likes"]:,} / コメ:{v["comments"]:,}\n'
                f'   タグ: {tags_str}\n'
                f'   説明: {v["description"][:100]}'
            )
        yt_data_text = '\n'.join(analysis_lines)
    else:
        yt_data_text = '\n'.join([f'{i+1}. 「{v["title"]}」' for i, v in enumerate(yt_basic)])
else:
    print('  ⚠ YouTube API取得できず。AI知識でトレンド分析します。')
    yt_data_text = f'テーマ「{THEME}」の一般的なトレンド情報'

# ── ③ Claude でトレンド＆音声スタイルを分析 ─────────────────
print(f'\n🤖 Claude AIでトレンドと音声スタイルを分析中...')
trend = call_ai(
    f'YouTubeの「{THEME}」直近90日のトレンドShortsデータ（実測値）:\n{yt_data_text}\n\n'
    f'このデータから以下を箇条書きで抽出してください:\n\n'
    f'【A. 今バズるタイトル公式】\n'
    f'・数字・固有名詞・「〜するな」「知らないと損」など煽り語の組み合わせパターン\n'
    f'・実際に高再生を記録しているタイトルの言葉遣い（3〜5例）\n\n'
    f'【B. 冒頭3秒フック（具体フレーズ）】\n'
    f'・視聴者が離脱しないために使うべき具体的な日本語フレーズ（5例）\n\n'
    f'【C. 今ウケるコンテンツの穴（ライバル動画が扱っていないネタ）】\n'
    f'・上位動画が触れていない切り口やアングル\n\n'
    f'【D. 視聴者が実際にコメントしそうな議論ネタ】\n'
    f'・「それ違う」「私もやってみた」が生まれやすい断言・比較・ランキング系テーマ\n\n'
    f'【E. 音声スタイル】語尾・テンポ・間の特徴（2〜3行）\n'
    f'日本語・簡潔に出力。'
)
print('  ✓ トレンド・音声分析完了')
print(f'\n  分析結果の一部:\n  {trend[:200]}...')

# ── ④ VIDEO_COUNT個の切り口を生成 ────────────────────────────
print(f'\n💡 切り口を{VIDEO_COUNT}個考案中...')
angles_raw = call_ai(
    f'テーマ「{THEME}」のYouTube Shortsタイトルを{VIDEO_COUNT}個出力してください。\n'
    f'トレンド分析: {trend[:500]}\n\n'
    f'【絶対守ること】\n'
    f'・タイトルのみ出力（説明・番号・記号なし）\n'
    f'・1行1タイトル\n'
    f'・数字を入れる（例: 3つの方法、1週間で）\n'
    f'・煽り系ワード（知らないと損、衝撃、やばい、本当は）\n'
    f'・20文字以内\n\n'
    f'出力例:\n痩せない本当の理由3つ\n1週間で-3kgの食事法\n食べても太らない時間帯',
    tokens=1024
)
ANGLES = [clean_title(l) for l in angles_raw.strip().split('\n')
          if l.strip() and len(l.strip()) > 3][:VIDEO_COUNT]
while len(ANGLES) < VIDEO_COUNT:
    ANGLES.append(f'{THEME}の秘密 Vol.{len(ANGLES)+1}')

# トレンド分析結果をセル4で使えるようにグローバル変数に保存
TREND_ANALYSIS = trend

print(f'\n生成する{VIDEO_COUNT}本:')
for i, a in enumerate(ANGLES):
    print(f'  {i+1:2d}. {a}')
print(f'\n✅ 切り口の決定完了')

In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）

import cv2, numpy as np, subprocess, tempfile, time, wave, json as _json, shutil as _shutil
import scipy.io.wavfile as wio
from pathlib import Path
from gtts import gTTS

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],
                       capture_output=True,text=True,timeout=300)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def get_wav_dur(path):
    try:
        with wave.open(str(path),'r') as wf:
            return wf.getnframes()/wf.getframerate()
    except:
        return 3.0

def clean_text(t):
    t = re.sub(r'#\S+', '', t)
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)|シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    t = re.sub(r'^#+\s*','',t); t = re.sub(r'[\*_]','',t)
    return re.sub(r'\s+',' ',t).strip()

def tts_text_cleanse(text):
    """TTS専用テキスト補正: gTTSが自然なイントネーションで読めるよう意味・語尾を修正"""
    # ① AI特有の語尾をgTTSが自然に読める語尾に変換
    text = re.sub(r'んだよ([！！]?)', r'よ\1', text)       # 〜んだよ！ → 〜よ！
    text = re.sub(r'んだよね', 'よね', text)
    text = re.sub(r'んだね([！！]?)', r'だよね\1', text)
    text = re.sub(r'んです([よ！ね。]?)', r'よ\1', text)    # 〜んです → 〜よ
    text = re.sub(r'なんだ([！！。]?)', r'だよ\1', text)   # 〜なんだ！ → 〜だよ！
    text = re.sub(r'なんです', 'だよ', text)
    text = re.sub(r'するんだ([！！。]?)', r'するよ\1', text)
    text = re.sub(r'できるんだ([！！。]?)', r'できるよ\1', text)
    text = re.sub(r'整うんだ([！！。]?)', r'整うよ\1', text)   # ユーザー指定例
    text = re.sub(r'変わるんだ([！！。]?)', r'変わるよ\1', text)  # ユーザー指定例
    text = re.sub(r'あるんだ([！！。]?)', r'あるよ\1', text)
    text = re.sub(r'いるんだ([！！。]?)', r'いるよ\1', text)
    # ② 読点（ポーズ）挿入: gTTSが自然な間を置けるよう感嘆詞の後に読点
    text = re.sub(r'^(え)(マジ|本当|すごい|やば)', r'\1、\2', text)  # えマジ→え、マジ
    text = re.sub(r'^(実は)([^、])', r'\1、\2', text)               # 実は→実は、
    text = re.sub(r'^(ねえ|ねっ|あのね)([^、])', r'\1、\2', text)   # ねえ→ねえ、
    text = re.sub(r'^(知ってる)(\?|？)?([^、])', r'\1？\3', text)
    # ③ 数字＋単位の間にスペース: gTTSが分けて読めるよう
    text = re.sub(
        r'([一二三四五六七八九十百千万億])(キロ|グラム|センチ|メートル|ヶ月|ヵ月|カ月|週間|日間|時間)',
        r'\1 \2', text
    )
    # ④ 重複記号を整理
    text = re.sub(r'！{2,}', '！', text)
    text = re.sub(r'、{2,}', '、', text)
    return text.strip()

def preprocess_tts(text):
    """ハッシュタグ・記号を完全除去し gTTS が読めるテキストへ"""
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[\[\](){}@&*|\\^~`<>=+_・]', '', text)
    text = re.sub(r'[^ -~　-鿿！-｠゠-ヿ぀-ゟ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 16 and '、' not in text:
        mid = len(text) // 2
        for j in range(max(0, mid-4), min(len(text)-1, mid+5)):
            if text[j] in 'はがをにでもよねてし':
                text = text[:j+1] + '、' + text[j+1:]
                break
    return text.strip() or 'つぎのシーンです'

def make_tts_voicevox(text, tmp_dir, prefix, speaker=None):
    """VOICEVOX REST APIで高品質日本語音声を生成（ずんだもんノーマル）"""
    if speaker is None: speaker = VV_SPEAKER
    try:
        import requests as _r
        # Step1: audio_query
        q = _r.post(f'{VV_URL}/audio_query',
                    params={'text': text, 'speaker': speaker}, timeout=30)
        q.raise_for_status()
        query = q.json()
        # 話速・イントネーション・無音長を調整
        query['speedScale']       = 1.05   # わずかに速め（gTTSより自然）
        query['intonationScale']  = 1.15   # イントネーション強調
        query['prePhonemeLength'] = 0.05   # 発話前の無音
        query['postPhonemeLength']= 0.10   # 発話後の無音
        # Step2: synthesis
        s = _r.post(f'{VV_URL}/synthesis',
                    params={'speaker': speaker}, json=query, timeout=60)
        s.raise_for_status()
        out = tmp_dir / f'{prefix}_vv.wav'
        out.write_bytes(s.content)
        return out
    except Exception as _e:
        print(f'  ⚠ VOICEVOX TTS失敗: {_e}')
        return None

def make_tts_audio(text, tmp_dir, prefix):
    """VOICEVOX優先、失敗時はgTTSフォールバックで音声生成"""
    text = preprocess_tts(text)
    # ── VOICEVOX優先ルート ───────────────────────────────────
    if VV_AVAILABLE:
        _vv = make_tts_voicevox(text, tmp_dir, prefix)
        if _vv and _vv.exists():
            return _vv
    # ── gTTSフォールバック ────────────────────────────────────
    sr = 44100
    # 句読点の後で分割（句読点を各フレーズに含める）
    parts = [p.strip() for p in re.split(r'(?<=[。！？、])', text) if p.strip() and len(p.strip()) >= 2]
    if not parts:
        parts = [text] if text else []
    if not parts:
        return None

    wavs = []
    for j, phrase in enumerate(parts):
        mp3 = tmp_dir/f'{prefix}_{j}.mp3'
        wav = tmp_dir/f'{prefix}_{j}.wav'
        sil = tmp_dir/f'{prefix}_{j}_s.wav'
        try:
            gTTS(text=phrase, lang='ja', slow=False).save(str(mp3))
            ff('-i',str(mp3),'-filter:a',
               'silenceremove=start_periods=1:start_silence=0.02:start_threshold=-42dB'
               ':stop_periods=-1:stop_silence=0.04:stop_threshold=-42dB,atempo=1.35',
               '-ar','44100','-ac','1',str(wav))
            wavs.append(wav)
            # 句点・感嘆符・疑問符 = 180ms、読点 = 80ms の無音
            sil_ms = 140 if phrase[-1] in '。！？' else 60
            wio.write(str(sil), sr, np.zeros(int(sr*sil_ms/1000), dtype=np.float32))
            wavs.append(sil)
        except:
            pass

    if not wavs:
        return None

    out = tmp_dir/f'{prefix}_out.wav'
    if len(wavs) == 1:
        _shutil.copy(str(wavs[0]), str(out))
    else:
        lf = tmp_dir/f'{prefix}_lf.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(out))
    return out

def wrap_subtitle(text, max_chars=9):
    text = re.sub(r'#\S+', '', text).strip()
    KINSOKU = '！？。、…〕）」』'  # 行頭禁則: これで行を始めてはいけない文字
    if len(text) > max_chars:
        # 自然な区切り位置を探す（句読点・助詞の直後）
        break_pos = None
        for i in range(max_chars, min(max_chars + 6, len(text))):
            if text[i] in '、。！？はがをにでもよね':
                break_pos = i + 1
                break
        if break_pos is None:
            break_pos = max_chars
        # 禁則処理: 改行直後が禁則文字なら前行に吸収する
        while break_pos < len(text) and text[break_pos] in KINSOKU:
            break_pos += 1
        text = text[:break_pos] + r'\N' + text[break_pos:]
    # ①②③④⑤を白色でハイライト（ASS インライン override）
    text = re.sub(r'([①②③④⑤⑥⑦⑧])', r'{\\c&H00FFFFFF&}\1{\\c&H0000FFFF&}', text)
    return text

def clean_subtitle_text(text):
    """字幕クレンジング: 句読点（。、）・冒頭感嘆詞を正規表現で除去"""
    text = re.sub(r'[。、]', '', text)
    text = re.sub(r'^(えっ|あっ|ねえ|ねっ|うわ|おっ|へえ|わあ|まあ|はあ|は)[！？!?]?\s*', '', text)
    return text.strip()

def auto_fit_fontsize(text, base=75, min_fs=48):
    """文字数に応じてフォントサイズを自動縮小（字幕エリア 55%-70% に収める）"""
    # ASS override タグを除去してから文字数カウント
    clean = re.sub(r'\{[^}]*\}', '', text).replace('\\N', '').replace('\\n', '')
    n = len(clean)
    if n <= 9:  return base        # 1行: 75pt
    if n <= 18: return 65          # 2行: 65pt
    return min_fs                  # 長文: 48pt

def generate_bgm(duration_sec, sr=44100):
    """128BPM アップテンポBGM（ピーク正規化済み・amixで音量制御）"""
    n = int(sr * duration_sec)
    bgm = np.zeros(n, dtype=np.float32)
    bpm = 128; beat = 60.0 / bpm
    scale = [261.63, 293.66, 329.63, 392.00, 440.00, 523.25, 587.33, 659.25]
    pattern = [0, 2, 4, 5, 4, 2, 0, 2, 4, 7, 6, 5, 4, 2, 0, 4]
    for i in range(int(duration_sec / beat) + len(pattern) + 1):
        pi = i % len(pattern); start = int(i * beat * sr)
        if start >= n: break
        end = min(int((i * beat + beat * 0.75) * sr), n); sz = end - start
        if sz <= 0: continue
        nt = np.linspace(0, sz/sr, sz); freq = scale[pattern[pi] % len(scale)]
        wav = (np.sin(2*np.pi*freq*nt)*0.5 + np.sin(2*np.pi*freq*2*nt)*0.15 + np.sin(2*np.pi*freq*3*nt)*0.08)
        env = np.ones(sz); a, r2 = min(int(0.01*sr), sz), min(int(0.06*sr), sz)
        env[:a] = np.linspace(0, 1, a); env[-r2:] *= np.linspace(1, 0, r2)
        bgm[start:end] += wav * env * 0.22
    for bi in range(int(duration_sec / beat) + 1):
        p = int(bi * beat * sr)
        if p >= n: break
        kl = min(int(0.07*sr), n-p)
        if kl <= 0: continue
        kt = np.linspace(0, 0.07, kl)
        bgm[p:p+kl] += (np.sin(2*np.pi*(90-60*kt/0.07)*kt) * np.exp(-kt*22) * (0.55 if bi%2==0 else 0.22))
    hh_step = beat / 2; rng = np.random.default_rng(42)
    for hi in range(int(duration_sec / hh_step) + 1):
        hp = int(hi * hh_step * sr)
        if hp >= n: break
        hl = min(int(0.018*sr), n-hp)
        if hl <= 0: continue
        ht = np.linspace(0, 0.018, hl)
        bgm[hp:hp+hl] += rng.standard_normal(hl) * np.exp(-ht*90) * 0.07
    bass = [130.81, 130.81, 196.00, 146.83]; bar = beat * 4
    for bi in range(int(duration_sec / bar) + 2):
        p = int(bi * bar * sr)
        if p >= n: break
        bl = min(int(bar*0.88*sr), n-p)
        if bl <= 0: continue
        bt = np.linspace(0, bl/sr, bl)
        bgm[p:p+bl] += np.sin(2*np.pi*bass[bi%len(bass)]*bt) * np.exp(-bt*0.8) * 0.38
    mx = np.max(np.abs(bgm))
    if mx > 0: bgm = bgm / mx
    return bgm

def make_cover_crop_vf(iw2, ih2):
    """入力サイズ → 9:16カバークロップ用 ffmpegフィルタ文字列。黒帯ゼロ保証。"""
    # COVER: AR保持のまま両辺がW/H以上になる最小スケール（コンマなし・パース安全）
    if iw2 * H > ih2 * W:       # 横長素材 → 高さ基準でスケール
        sw = iw2 * H // ih2; sh = H
    else:                        # 縦長素材 → 幅基準でスケール
        sw = W; sh = ih2 * W // iw2
    cx2 = (sw - W) // 2; cy2 = (sh - H) // 2  # 中心基準クロップ座標
    return f'scale={sw}:{sh},crop={W}:{H}:{cx2}:{cy2}'

def make_clip(img_path, dur, out, idx=0, boost=1.0):
    """静止画クリップ: 全シーン動的 Ken Burns（1.05→1.25ズーム・boost でジャンプカット対応）"""
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.05 * boost:.4f}"  # zoom-in 開始倍率
    ZE = f"{1.25 * boost:.4f}"  # zoom-in 終了倍率 / zoom-out 開始倍率
    ZD = f"{0.20 * boost:.4f}"  # ズーム変化幅
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    styles = [
        # 0: zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # 1: zoom-in(1.05→1.25) + 下パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))/2", f"(ih-ih/({ZS}+{ZD}*n/{N}))*n/{N}"),
        # 2: zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
        # 3: zoom-out(1.25→1.05) + 上パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))/2", f"(ih-ih/({ZE}-{ZD}*n/{N}))*(1-n/{N})"),
    ]
    cw, ch, cx, cy = styles[idx % len(styles)]
    if Path(img_path).exists():
        # 9:16カバークロップ: 画像の実寸をプローブして黒帯ゼロを保証
        try:
            _pb = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(img_path)],
                                 capture_output=True, text=True, timeout=10)
            _vs = next((s for s in _json.loads(_pb.stdout).get('streams',[])
                        if s.get('codec_type')=='video'), None)
            _iw, _ih = (int(_vs['width']), int(_vs['height'])) if _vs else (W, H)
        except:
            _iw, _ih = W, H
        cover = make_cover_crop_vf(_iw, _ih)
        vf = f"{cover},crop={cw}:{ch}:{cx}:{cy},scale={W}:{H}"
        ff('-loop','1','-i',str(img_path),'-vf',vf,
           '-t',str(dur),'-r',str(FPS),'-an',
           '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p',str(out))
    else:
        ff('-f','lavfi','-i',f'color=c=black:s={W}x{H}:r={FPS}',
           '-t',str(dur),'-c:v','libx264','-preset','ultrafast',str(out))

def fetch_pexels_video(kw):
    """Pexelsから縦向き12秒以内の動画URLを取得"""
    try:
        r = _req.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': PEXELS_API_KEY},
                     params={'query': kw, 'per_page': 5, 'orientation': 'portrait'},
                     timeout=15)
        if r.status_code != 200: return None, 0
        for v in r.json().get('videos', []):
            vdur = v.get('duration', 99)
            if vdur > 12: continue
            files = sorted(v.get('video_files', []), key=lambda x: x.get('height', 0), reverse=True)
            for f in files:
                if f.get('height', 0) >= 480: return f['link'], vdur
    except:
        pass
    return None, 0

def make_clip_from_video(video_path, dur, out, idx=0, boost=1.0):
    """Pexels動画を縦型1080x1920にクロップ・ループ変換（動的 Ken Burns 付き）"""
    probe = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(video_path)],
                           capture_output=True, text=True)
    try:
        vs = next((s for s in _json.loads(probe.stdout).get('streams',[]) if s.get('codec_type')=='video'), None)
        vw, vh = (int(vs['width']), int(vs['height'])) if vs else (W, H)
    except:
        vw, vh = W, H
    cover = make_cover_crop_vf(vw, vh)  # 9:16カバークロップ（黒帯ゼロ・中心基準）
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.05 * boost:.4f}"  # zoom-in 開始倍率
    ZE = f"{1.25 * boost:.4f}"  # zoom-in 終了倍率 / zoom-out 開始倍率
    ZD = f"{0.20 * boost:.4f}"  # ズーム変化幅
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    kb = [
        # zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
    ]
    kcw, kch, kcx, kcy = kb[idx % len(kb)]
    ff('-stream_loop','-1','-i',str(video_path),
       '-vf',f'{cover},crop={kcw}:{kch}:{kcx}:{kcy},scale={W}:{H}',
       '-t',str(dur),'-r',str(FPS),'-an',
       '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p',str(out))

W, H, FPS = 1080, 1920, 30
# VOICEVOX定数（セル3で設定済みの値を継承、未実行時はデフォルト）
try: VV_AVAILABLE
except NameError: VV_AVAILABLE = False
try: VV_URL
except NameError: VV_URL = 'http://127.0.0.1:50021'
try: VV_SPEAKER
except NameError: VV_SPEAKER = 3
VIDEO_LEAD = 0.05  # シーン2以降: 映像カットが音声より50ms先行
# 動画を使うシーンのインデックス（0始まり）: フック・3番目・5番目
VIDEO_SCENE_IDX = {0, 2, 4}
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
completed = []
failed = []

def rewrite_script_to_natural(scene_texts, theme, angle):
    """Step2: AI生成台本を日本人女性インフルエンサーの自然口語に完全リライト"""
    if not scene_texts:
        return scene_texts
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        rewritten = call_ai(
            f'あなたは日本の人気女性YouTuberです。以下の台本を、スマホカメラに向かって'
            f'実際に話しているような100%自然な口語にリライトしてください。\n\n'
            f'【元の台本】\n{scenes_str}\n\n'
            f'【テーマ】{theme}  【タイトル】{angle}\n\n'
            f'【絶対に守るルール】\n'
            f'・「〜なんだ」「〜なの？」「〜するんだよ」「〜んです」など\n'
            f'  ロボット調・直訳調の語尾は完全に禁止\n'
            f'・語尾は「〜だよ！」「〜よね！」「〜してみて！」「〜のがポイント！」\n'
            f'  「〜じゃない？」など女性インフルエンサーが実際に使う語尾に統一\n'
            f'・①②③番号付きtipsは「〜する！」「〜を選ぶ！」の言い切り型に変換\n'
            f'・シーン1は「え、マジ？」「これ知ってた？」「実はね、」のような\n'
            f'  視聴者が思わず手を止める自然なフックで始める\n'
            f'・各セリフは10〜14文字の短いひとこと（長い文は絶対NG）\n'
            f'・ひらがな・カタカナ・漢字のみ（英語・記号・ハッシュタグ完全禁止）\n'
            f'・【コメント誘発】全シーンの中に1〜2箇所、視聴者が思わず突っ込むフレーズを入れる\n'
            f'  （例：「〜は絶対やるな！」「知らないと一生損する」「〜したら人生変わった」）\n\n'
            f'【出力フォーマット（厳守）】\n'
            f'セリフのみを出力。説明・コメント・空行は一切不要。\n'
            f'シーン1: （リライト後のセリフのみ）\n'
            f'シーン2: （リライト後のセリフのみ）\n'
            f'（元と同じシーン数で出力すること）',
            tokens=1024
        )
        # リライト結果をシーン単位でパース
        result = []
        for line in rewritten.strip().split('\n'):
            m = re.match(r'シーン\d+[:：]\s*(.+)', line.strip())
            if m:
                t = clean_text(m.group(1).strip())
                if t: result.append(t)
        # シーン数が一致しない場合は元テキストで補完
        while len(result) < len(scene_texts):
            result.append(scene_texts[len(result)])
        return result[:len(scene_texts)]
    except Exception as e:
        print(f'  ⚠ リライト失敗（元テキストにフォールバック）: {e}')
        return scene_texts

def make_one_video(idx, angle):
    vid_num = idx + 1
    safe_angle = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    print(f'\n{"─"*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print(f'{"─"*50}')
    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    MEDIA = TMP/'media'; MEDIA.mkdir()

    # ── 台本生成（シーン数はClaudeが内容に応じて自動決定） ────
    print('  📝 台本生成中...')
    _trend_ctx = globals().get('TREND_ANALYSIS', '')[:600]
    _trend_block = (
        f'【最新トレンド分析（実際のYouTube上位動画から抽出）】\n{_trend_ctx}\n\n'
        if _trend_ctx else ''
    )
    raw = call_ai(
        f'{_trend_block}'
        f'YouTube Shorts台本を作成してください。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n'
        f'【目標秒数】{DURATION}秒前後（15〜60秒の範囲）\n\n'
        f'【シーン数の決め方（重要）】\n'
        f'・シンプルな1ポイントtips → 3〜4シーン\n'
        f'・方法・コツ系 → 5〜6シーン\n'
        f'・詳しい解説系 → 7〜8シーン\n'
        f'内容の複雑さに合わせて自由にシーン数を決めてください\n\n'
        f'【絶対ルール】\n'
        f'・ハッシュタグ（#）・記号（@&*[]等）・英単語・URL完全禁止\n'
        f'・ひらがな・カタカナ・漢字のみ\n'
        f'・各セリフは10〜18文字の短いひとこと\n'
        f'・感情の起伏（驚き→共感→解決→期待）\n'
        f'・語尾: 〜だよ！/〜してみて/〜のがコツ/〜が大事/〜だよね/〜してみよう\n'
        f'・「〜なの？」「〜むんだよ」など直訳風・不自然な語尾は絶対NG\n'
        f'・読点（、）で自然な間（例：実は、知ってた？/毎朝、やってみて！）\n'
        f'・シーン1は衝撃フック（これ知ってた？/知らないと損！/えっ本当に？ 等）\n'
        f'・①②③の番号付きtipsは必ず言い切り型「〜する！」（例:①白湯を朝に飲む！②食前に一杯！）\n'
        f'・【コメント誘発ルール】全シーンの中に1〜2箇所、視聴者がコメント欄で突っ込みたくなる\n'
        f'  「極端な断言・煽り」を入れる（例:「〜は絶対やるな！」「知らないと一生損する」\n'
        f'  「これやめたら人生変わった」「〜してる人は今すぐやめて」）\n\n'
        f'【キーワードルール（重要）】\n'
        f'・各シーンで映すべき具体的な映像を英語1〜2語で指定する\n'
        f'・NG例:「lifestyle」「woman」「person」\n'
        f'・OK例:「healthy food」「warm water drink」「grilled chicken」「bedroom clock」\n'
        f'・シーン1（フック）は必ず料理・食材・具体的な行動が映るキーワード\n\n'
        f'[台本]\n'
        f'シーン1（フック）: \n'
        f'シーン2（共感）: \n'
        f'シーン3（①1つ目 or 解説1）: \n'
        f'（以降、内容に応じたシーン数まで続ける）\n\n'
        f'[キーワード]\n'
        f'scene1: （料理・食材など具体的な英語1〜2語）\n'
        f'（台本と同じシーン数）'
    )

    script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw) or
                   type('x',(),{'group':lambda s,n:raw})()).group(1)
    kw_block = re.search(r'\[キーワード\]([\s\S]*?)$', raw)

    # 実際に生成されたシーン数をカウント（動的SCENE_COUNT）
    scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
    scene_texts = [clean_text(s) for s in scene_lines[1:] if s.strip()]
    SCENE_COUNT = max(3, min(8, len(scene_texts)))
    while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)
    scene_texts = scene_texts[:SCENE_COUNT]

    keywords = []
    if kw_block:
        for line in kw_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m: keywords.append(m.group(1).strip())
    while len(keywords) < SCENE_COUNT: keywords.append('lifestyle')
    keywords = keywords[:SCENE_COUNT]
    print(f'  ✓ Step1 台本生成完了（{SCENE_COUNT}シーン）')

    # ── Step 2: 日本語校正 & SNS自然口語リライト ──────────────
    print('  ✏️  Step2: 自然口語リライト中（インフルエンサー調）...')
    scene_texts = rewrite_script_to_natural(scene_texts, THEME, angle)
    if scene_texts:
        print(f'  ✓ リライト完了 scene1→ {scene_texts[0]}')
    # TTS用テキスト（語尾・間補正済み）を字幕テキストと分離して生成
    tts_texts = [tts_text_cleanse(t) for t in scene_texts]

    # ── メディア取得（動画と静止画を明示的に組み合わせ） ─────
    print(f'  🖼 メディア取得中（動画シーン:{sorted(VIDEO_SCENE_IDX)}、他は静止画）...')
    scenes = []
    for i, kw in enumerate(keywords):
        time.sleep(0.7)
        img_fn = MEDIA/f'img_{i+1:02d}.jpg'
        vid_fn = MEDIA/f'vid_{i+1:02d}.mp4'
        is_video = False

        # VIDEO_SCENE_IDXのシーンのみ動画を試みる
        if i in VIDEO_SCENE_IDX:
            # フックシーン(i==0)は複数キーワードで動画検索（フォールバックあり）
            kw_candidates = [kw, 'food close up', 'healthy meal', 'delicious food'] if i == 0 else [kw]
            for try_kw in kw_candidates:
                vid_url, _ = fetch_pexels_video(try_kw)
                if vid_url:
                    try:
                        dl = _req.get(vid_url, timeout=60)
                        if dl.status_code == 200:
                            vid_fn.write_bytes(dl.content)
                            is_video = True
                            break
                    except:
                        pass
                if is_video: break

        # 静止画取得（動画なし or VIDEO_SCENE_IDX外）
        if not is_video:
            try:
                r = _req.get('https://api.pexels.com/v1/search',
                             headers={'Authorization': PEXELS_API_KEY},
                             params={'query': kw, 'per_page': 3, 'orientation': 'portrait'}, timeout=15)
                if r.status_code == 200:
                    photos = r.json().get('photos', [])
                    if photos:
                        photo = photos[i % len(photos)]
                        url = photo['src'].get('portrait') or photo['src'].get('large')
                        dl = _req.get(url, timeout=30)
                        if dl.status_code == 200: img_fn.write_bytes(dl.content)
            except Exception as e:
                print(f'  ⚠ 画像スキップ scene{i+1}: {e}')

        tag = '🎬動画' if is_video else '🖼静止画'
        print(f'    scene{i+1}: {tag} [{kw}]')
        scenes.append({'scene':i+1,'image':img_fn,'video':vid_fn,'is_video':is_video,
                       'keyword':kw,
                       'text': scene_texts[i],   # 字幕表示用（クリーンな短文）
                       'tts':  tts_texts[i]})
    print('  ✓ メディア取得完了')

    # ── スタイル変換（静止画のみ） ────────────────────────────
    if IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            if not s['is_video'] and s['image'].exists():
                img = cv2.imread(str(s['image']))
                if img is not None: cv2.imwrite(str(s['image']), fn_style(img))
        print('  ✓ スタイル変換完了')

    # ── 音声生成（フレーズ分割・句読点ごとに自然な間） ────────
    print('  🎙 音声生成中（フレーズ分割・自然な間）...')
    wavs = []
    for i, s in enumerate(scenes):
        wav_out = make_tts_audio(s.get('tts') or s['text'] or THEME, TMP, f'tts{i}')
        if wav_out and wav_out.exists():
            audio_dur = get_wav_dur(wav_out)
            s['duration'] = max((0.0 if i == 0 else VIDEO_LEAD) + audio_dur, 1.5)
            wavs.append(wav_out)
        else:
            s['duration'] = max(DURATION / SCENE_COUNT, 2.0)
            print(f'  ⚠ 音声スキップ scene{i+1}')

    total_dur = sum(s['duration'] for s in scenes)
    cap_dur = min(max(total_dur, 15.0), 60.0)  # YouTube Shorts: 15〜60秒
    print(f'  ✓ 音声完了 合計{total_dur:.1f}秒 → 出力{cap_dur:.0f}秒')

    # ── 動画クリップ生成 ──────────────────────────────────────
    print('  🎬 クリップ生成中（3秒超えシーン自動ジャンプカット分割）...')
    SPLIT_THRESHOLD = 3.0  # これ以上の長さのシーンは2分割してテンポ感を生成
    clips = []
    for i_c, s in enumerate(scenes):
        d = s['duration']
        has_vid = s['is_video'] and s['video'].exists()

        def _mk(dur, out_path, kb_idx, boost=1.0):
            """ソース種別に応じたクリップ生成（boost=1.0 通常 / 1.2 ジャンプカット後半）"""
            if has_vid:
                try:
                    make_clip_from_video(s['video'], dur, out_path, idx=kb_idx, boost=boost)
                    return
                except Exception as e:
                    print(f'  ⚠ 動画変換失敗→静止画: {e}')
            make_clip(s['image'], dur, out_path, idx=kb_idx, boost=boost)

        if d > SPLIT_THRESHOLD:
            # 3秒超え: 前半（通常）+ 後半（1.2倍ズームアップ）で疑似ジャンプカット
            half = d / 2
            out1 = TMP/f"c{s['scene']:02d}a.mp4"
            out2 = TMP/f"c{s['scene']:02d}b.mp4"
            _mk(half, out1, i_c, boost=1.0)
            _mk(half, out2, i_c + 1, boost=1.2)
            clips.extend([out1, out2])
            print(f"  ✂ scene{s['scene']} ジャンプカット分割 ({half:.1f}s × 2 / boost×1.2)")
        else:
            out = TMP/f"c{s['scene']:02d}.mp4"
            _mk(d, out, i_c, boost=1.0)
            clips.append(out)

    lf2 = TMP/'cl.txt'
    lf2.write_text('\n'.join(f"file '{p}'" for p in clips))
    mg = TMP/'m.mp4'
    ff('-f','concat','-safe','0','-i',str(lf2),'-c','copy',str(mg))

    # ── 字幕（黄色・大きめ） ─────────────────────────────────
    ah = (
        f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
        f"[V4+ Styles]\n"
        f"Format:Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,"
        f"Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,"
        f"Alignment,MarginL,MarginR,MarginV,Encoding\n"
        f"Style:Default,Noto Sans CJK JP,75,&H0000FFFF,&H000000FF,&H00000000,&H80000000,"
        f"-1,0,0,0,100,100,2,0,3,12,0,2,{int(W*0.10)},{int(W*0.10)},{int(H*0.28)},1\n"
        f"Style:Hook,Noto Sans CJK JP,76,&H00FFFFFF,&H000000FF,&H00000000,&H000000FF,"
        f"-1,0,0,0,100,100,0,0,3,20,0,8,60,60,{int(H*0.05)},1\n\n"
        f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"
    )
    ev = []; t = 0.0
    # ── 冒頭フックテロップ（0〜2秒・Hook スタイル・赤背景）───────────
    if '】' in angle and '【' in angle:
        _hl1 = angle[:angle.index('】')+1]; _hl2 = angle[angle.index('】')+1:].strip()
        _htxt = _hl1 + r'\N' + _hl2 if _hl2 else _hl1
    elif len(angle) > 10:
        _m = len(angle)//2; _htxt = angle[:_m] + r'\N' + angle[_m:]
    else:
        _htxt = angle
    _hend = min(2.0, scenes[0]['duration'] if scenes else 2.0)
    ev.append(f"Dialogue:0,{at(0)},{at(_hend)},Hook,,0,0,0,,{_htxt}")
    # 字幕配置: 画面高さ55%-70%エリア強制（n5\pos で中心Y=62%固定）
    SUB_CX = W // 2
    SUB_CY = int(H * 0.62)  # 55%-70%エリアの中心 (Y軸)
    for s in scenes:
        sub_text = wrap_subtitle(clean_subtitle_text(s['text']))
        fs = auto_fit_fontsize(sub_text)
        # テロップは映像クリップ境界の1フレーム前に終了（重なりバグ排除）
        sub_end = t + s['duration'] - 1.0/FPS
        ev.append(f"Dialogue:0,{at(t)},{at(max(t+0.1, sub_end))},Default,,0,0,0,,"
                  f"{{\\an5\\pos({SUB_CX},{SUB_CY})\\fs{fs}}}{sub_text}")
        t += s['duration']
    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    # ── ナレーション音声を結合 ────────────────────────────────
    comb = TMP/'vc.wav'; vaac = TMP/'v.aac'
    # シーン1以降の先頭に VIDEO_LEAD 分の無音を挿入（映像50ms先行同期）
    _aparts = []
    for _iw, _wf in enumerate(wavs):
        if _iw > 0:
            _ls = TMP/f'_ls{_iw}.wav'
            wio.write(str(_ls), 44100, np.zeros(int(44100*VIDEO_LEAD), dtype=np.float32))
            _aparts.append(_ls)
        _aparts.append(_wf)
    if _aparts:
        lf = TMP/'vl.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in _aparts))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(comb))
    else:
        wio.write(str(comb), 44100, np.zeros(int(44100*total_dur), dtype=np.float32))
    ff('-i',str(comb),'-c:a','aac','-ar','44100',str(vaac))

    # ── BGM生成・ノイズフロア生成・最終合成 ─────────────────────
    bgm_path = TMP/'bgm.wav'
    wio.write(str(bgm_path), 44100, generate_bgm(cap_dur + 2))
    # Lo-Fiノイズフロア: ぶつ切り無音を聴覚的に隠蔽（約-30dB）
    noise_path = TMP/'noise.wav'
    _nsr = 44100; _nn = int(_nsr * (cap_dur + 2))
    _rng = np.random.default_rng(42)
    _raw = _rng.standard_normal(_nn).astype(np.float32)
    # 簡易ローパス（Lo-Fi感: 2kHz以下）: 16サンプル移動平均
    _k = np.ones(16, dtype=np.float32) / 16
    _smooth = np.convolve(_raw, _k, mode='same')
    _smooth = (_smooth / (np.abs(_smooth).max() + 1e-9) * 0.03).astype(np.float32)
    wio.write(str(noise_path), _nsr, _smooth)
    print('  🎵 BGM + ノイズフロア完了')

    # ── シームレスループ: 実際のナレーション尺を計測 ──────────
    try:
        _pb = subprocess.run(
            ['ffprobe','-v','quiet','-show_entries','format=duration',
             '-of','default=noprint_wrappers=1:nokey=1',str(vaac)],
            capture_output=True, text=True, timeout=10)
        _actual = float(_pb.stdout.strip())
        # 0.1秒トリム → 末尾の余白ゼロ → シームレスループ
        cap_dur = round(min(max(_actual - 0.1, 15.0), 60.0), 2)
        print(f'  ✂️  シームレスループ: {_actual:.2f}s → {cap_dur:.2f}s (0.1s trim)')
    except Exception as _pe:
        print(f'  ⚠ ループトリム計測スキップ: {_pe}')

    safe_theme_f = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    OUT = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_f}_{safe_angle}.mp4'

    ff('-i',str(mg),'-i',str(vaac),'-i',str(bgm_path),'-i',str(noise_path),
       '-filter_complex',
       '[1:a]volume=1.0[narr];'
       '[2:a]volume=0.65[bgm];'
       '[3:a]volume=1.0[nz];'
       '[narr][bgm][nz]amix=inputs=3:duration=first:normalize=0[aout]',
       '-vf',f'ass={se}',
       '-map','0:v','-map','[aout]',
       '-c:v','libx264','-preset','fast','-crf','20',
       '-c:a','aac','-b:a','192k',
       '-pix_fmt','yuv420p','-movflags','+faststart',
       '-t',str(cap_dur),str(OUT))

    if not OUT.exists() or OUT.stat().st_size < 10000:
        raise RuntimeError(f'動画生成失敗: {OUT}')

    mb = OUT.stat().st_size/1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f}MB, {cap_dur:.0f}秒)')
    return str(OUT)

start_all = time.time()
for idx, angle in enumerate(ANGLES):
    try:
        out_path = make_one_video(idx, angle)
        completed.append((idx+1, angle, out_path))
    except Exception as e:
        import traceback
        print(f'\n  ❌ エラー: {traceback.format_exc()[-500:]}')
        failed.append((idx+1, angle, str(e)))

elapsed = (time.time()-start_all)/60
print(f'\n{"="*50}')
print(f'🎉 完了！ 成功:{len(completed)}本 / 失敗:{len(failed)}本 / {elapsed:.1f}分')
print(f'{"="*50}')


In [ ]:
# 【自動】完成動画をダウンロード（触らなくてOK）
from IPython.display import Video, display, Audio
from google.colab import files
import shutil, numpy as np

print(f'✅ 完成した動画 ({len(completed)}本):')
for num, angle, path in completed:
    mb = Path(path).stat().st_size / 1_048_576
    print(f'  {num:2d}. {angle}  [{mb:.1f}MB]')

if failed:
    print(f'\n❌ 失敗 ({len(failed)}本):')
    for num, angle, err in failed:
        print(f'  {num:2d}. {angle} → {err[:80]}')

# プレビュー表示
if completed:
    print('\n▶ 1本目をプレビュー:')
    shutil.copy(completed[0][2], '/content/preview.mp4')
    display(Video('/content/preview.mp4', width=360))

# 完了通知音
sr = 44100
beep = np.sin(2*np.pi*880*np.linspace(0,0.3,int(sr*0.3)))*0.5
silent = np.zeros(int(sr*0.1))
display(Audio(np.concatenate([beep,silent,beep]), rate=sr, autoplay=True))

# ダウンロード
print('\n⬇️ 動画をダウンロードしています...')
for num, angle, path in completed:
    print(f'  ダウンロード中: {Path(path).name}')
    files.download(path)